In [3]:
"""
====================================================
  Calcul de la VaR par la méthode Cornish-Fisher
  Portefeuille d'actions (CAC 40)
====================================================
"""

import pandas as pd
import numpy as np
from scipy.stats import norm, kurtosis, skew

# ─────────────────────────────────────────────
# 1. CHARGEMENT DES DONNÉES
# ─────────────────────────────────────────────
df = pd.read_excel("Data_set.xlsx", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)
df.set_index("Date", inplace=True)

# ─────────────────────────────────────────────
# 2. RENDEMENTS LOGARITHMIQUES
# ─────────────────────────────────────────────
rendements = np.log(df / df.shift(1)).dropna()

# ─────────────────────────────────────────────
# 3. PORTEFEUILLE (poids égaux)
# ─────────────────────────────────────────────
n_actions = len(rendements.columns)
poids = np.array([1 / n_actions] * n_actions)
rend_port = rendements @ poids

# ─────────────────────────────────────────────
# 4. STATISTIQUES
# ─────────────────────────────────────────────
mu    = rend_port.mean()
sigma = rend_port.std()
S     = skew(rend_port)       # Asymétrie
K     = kurtosis(rend_port)   # Kurtosis en excès

# ─────────────────────────────────────────────
# 5. VaR CORNISH-FISHER (niveau 99%)
# ─────────────────────────────────────────────
alpha = 0.01
z = norm.ppf(alpha)

# Quantile ajusté Cornish-Fisher
z_CF = (z
        + (z**2 - 1) * S / 6
        + (z**3 - 3*z) * K / 24
        - (2*z**3 - 5*z) * S**2 / 36)

VaR_CF = -(mu + sigma * z_CF)

# ─────────────────────────────────────────────
# 6. INTERVALLE DE CONFIANCE PAR BOOTSTRAP
# ─────────────────────────────────────────────
np.random.seed(1234)
nb_simul   = 5000
boot_CF    = np.zeros(nb_simul)
rend_array = rend_port.values

for i in range(nb_simul):
    sample  = np.random.choice(rend_array, size=len(rend_array), replace=True)
    z_CF_b  = (z
               + (z**2 - 1) * skew(sample) / 6
               + (z**3 - 3*z) * kurtosis(sample) / 24
               - (2*z**3 - 5*z) * skew(sample)**2 / 36)
    boot_CF[i] = -(sample.mean() + sample.std() * z_CF_b)

ci_low  = np.quantile(boot_CF, 0.025)
ci_high = np.quantile(boot_CF, 0.975)


print(f"  Skewness            : {S:.4f}")
print(f"  Kurtosis (excès)    : {K:.4f}")
print(f"  Quantile CF ajusté  : {z_CF:.4f}")
print(f"  VaR CF 99% (1 jour) : {VaR_CF:.4%}")
print(f"  IC 95% Bootstrap    : [{ci_low:.4%} ; {ci_high:.4%}]")

  Skewness            : -0.3291
  Kurtosis (excès)    : 4.3844
  Quantile CF ajusté  : -3.5526
  VaR CF 99% (1 jour) : 3.7898%
  IC 95% Bootstrap    : [3.0321% ; 4.3982%]


In [ ]:

        
        # On examine plusieurs seuils candidats
        candidate_quantiles = [0.94, 0.95, 0.96, 0.97]

        for q in candidate_quantiles:
            u = losses.quantile(q)
            Nu = len(losses[losses > u])
            print(f"Quantile {q} → seuil {u:.5f} → excès {Nu} ({Nu/len(losses):.2%})")

        threshold = losses.quantile(0.95)
        excess = losses[losses > threshold] - threshold*

        # 4.3 Estimation GPD (MLE)
        xi, loc, beta = genpareto.fit(excess)

        params = pd.DataFrame({
            "Paramètre": ["xi (forme)", "beta (échelle)"],
            "Valeur": [xi, beta]
        })

        params

        # 4.4 Calcul de la VaR TVE à 99%
        # Nombre total d'observations
        n = len(losses)

        # Nombre d'excès
        Nu = len(excess)

        alpha = 0.99

        VaR_evt = threshold + (beta/xi) * (((n/Nu)*(1-alpha))**(-xi) - 1)

        print(f"VaR TVE à 99% : {VaR_evt:.4%}")

        # Calcul de la VaR normale à 99%
        from scipy.stats import norm

        mu = returns.mean()
        sigma = returns.std()

        VaR_norm = -(mu + sigma * norm.ppf(0.01))

        print(f"VaR Normale à 99% : {VaR_norm:.4%}")

        # TVE GARCH
        #5.1 Estimation GARCH(1,1) : On travaille sur les rendements
        model = arch_model(100*returns, vol='Garch', p=1, q=1)

        res = model.fit(disp="off")

        print(res.summary())

        #5.2 Extraction des Résidus standardisés
        std_resid = res.resid / res.conditional_volatility
        std_resid = std_resid.dropna()

        #Visualisation
        plt.figure(figsize=(10,4))
        plt.plot(std_resid**2)
        plt.title("Carrés des résidus standardisés")
        plt.show()

        #5.3 TVE sur résidus

        Z_losses = -std_resid # on travaille sur la queue gauche

        # CHOIX DU SEUIL
        threshold_z = Z_losses.quantile(0.95)

        excess_z = Z_losses[Z_losses > threshold_z] - threshold_z

        n_z = len(Z_losses)
        Nu_z = len(excess_z)

        print("Excès :", Nu_z)
        print("Proportion :", Nu_z/n_z)

        # Estimation de la GPD sur les résidus standardisés
        xi_z, loc_z, beta_z = genpareto.fit(excess_z)

        print(f"xi (résidus) : {xi_z:.4f}")
        print(f"beta (résidus) : {beta_z:.4f}")

     VaR_models = {

    "Historique": VaR_hist_series,
    "Paramétrique": VaR_norm_series,
    "Cornish-Fisher": VaR_cf_series,
    "RiskMetrics": VaR_rm_series,
    "GARCH": VaR_garch_dynamic,
    "TVE": VaR_evt_series,
    "TVE-GARCH": VaR_evt_garch_dynamic
}

    results = {}

for name, var in VaR_models.items():

    results[name] = backtest_var(returns, var)

# Tableau final

results_df = pd.DataFrame(results).T

results_df